# Data Preparation — Federated Mammography Pipeline
**AMS26-15 PMF** — Detekcija/Segmentacija tumora · Federativno učenje

This notebook:
1. Installs all dependencies
2. Clones the project from GitHub
3. Downloads each dataset from Kaggle
4. Converts all three datasets to a **unified annotation schema**
5. Validates and cleans every `annotations.csv`
6. Displays per-client statistics
7. Saves all client data to Google Drive

---
**Unified annotation schema** (binary classification: lesion or no lesion):

| Column | Type | Values |
|---|---|---|
| `image_name` | str | PNG filename |
| `bbox_xmin/ymin/width/height` | float | NaN for normal cases |
| `lesion_type` | str | `mass`, `calcification`, `distortion`, `asymmetry`, `normal`, `other` |
| `pathology` | str | `benign`, `malignant`, `probably_benign`, `normal`, `unknown` |
| `label` | int | **0** = no lesion, **1** = lesion present |
| `dataset_name` | str | `INbreast`, `CBIS-DDSM`, `MIAS` |

## 1 · Install dependencies

In [ ]:
!pip install -q pydicom opencv-python-headless openpyxl tqdm kaggle pandas numpy

## 2 · Clone project repository

In [ ]:
import os

REPO_URL  = 'https://github.com/MarijaGijic/Federated_project.git'
REPO_DIR  = '/content/federated_project'
DATA_DIR  = '/content/data'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR}/Federated_project pull
    print('Repo updated.')

os.makedirs(DATA_DIR, exist_ok=True)

import sys
PROJECT_ROOT = os.path.join(REPO_DIR, 'Federated_project')
sys.path.insert(0, PROJECT_ROOT)
scripts = os.path.join(PROJECT_ROOT, 'scripts')
print('Project root:', PROJECT_ROOT)

## 3 · Kaggle API setup
Upload your `kaggle.json` API token (from https://www.kaggle.com/settings → API → Create New Token).

In [ ]:
from google.colab import files

print('Upload kaggle.json:')
uploaded = files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API key installed.')

## 4 · Download datasets from Kaggle

In [ ]:
# ── INbreast ──────────────────────────────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/ramanathansp20/inbreast-dataset
INBREAST_RAW = '/content/raw/inbreast'
os.makedirs(INBREAST_RAW, exist_ok=True)
!kaggle datasets download -d ramanathansp20/inbreast-dataset -p {INBREAST_RAW} --unzip
print('INbreast downloaded.')

In [ ]:
# ── CBIS-DDSM ─────────────────────────────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset
CBIS_RAW = '/content/raw/cbis-ddsm'
os.makedirs(CBIS_RAW, exist_ok=True)
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset -p {CBIS_RAW} --unzip
print('CBIS-DDSM downloaded.')

In [ ]:
# ── MIAS ──────────────────────────────────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/kmader/mias-mammography
MIAS_RAW = '/content/raw/mias'
os.makedirs(MIAS_RAW, exist_ok=True)
!kaggle datasets download -d kmader/mias-mammography -p {MIAS_RAW} --unzip
print('MIAS downloaded.')

## 5 · Convert all datasets → unified PNG + annotations.csv
Each dataset becomes one "client" with the same directory layout:
```
data/
  client1_inbreast/
    images/          ← PNG images
    annotations.csv  ← canonical schema
  client2_cbisddsm/
  client3_mias/
```

In [ ]:
import glob

# ── Locate INbreast XLS ───────────────────────────────────────────────────────
xls_files = glob.glob(f'{INBREAST_RAW}/**/*.xls', recursive=True)
EXCEL_PATH = xls_files[0] if xls_files else ''
print('INbreast XLS:', EXCEL_PATH)

CLIENT1 = f'{DATA_DIR}/client1_inbreast'
!python {scripts}/prepare_inbreast.py \
    --raw_path    {INBREAST_RAW} \
    --output_path {CLIENT1} \
    --excel_path  {EXCEL_PATH}

In [ ]:
CLIENT2 = f'{DATA_DIR}/client2_cbisddsm'
!python {scripts}/prepare_cbisddsm.py \
    --raw_path    {CBIS_RAW} \
    --output_path {CLIENT2}

In [ ]:
CLIENT3 = f'{DATA_DIR}/client3_mias'
!python {scripts}/prepare_mias.py \
    --raw_path    {MIAS_RAW} \
    --output_path {CLIENT3}

## 6 · Validate and auto-clean all annotations

In [ ]:
!python {scripts}/validate_and_clean.py \
    --client_dirs {CLIENT1} {CLIENT2} {CLIENT3} \
    --fix

## 7 · Cross-dataset statistics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

clients = {
    'INbreast':  CLIENT1,
    'CBIS-DDSM': CLIENT2,
    'MIAS':      CLIENT3,
}

frames = []
for name, cdir in clients.items():
    ann = os.path.join(cdir, 'annotations.csv')
    if os.path.isfile(ann):
        frames.append(pd.read_csv(ann))
    else:
        print(f'WARNING: {ann} not found')

all_df = pd.concat(frames, ignore_index=True)
print(f'Total annotations: {len(all_df)}')
print(f'Unique images: {all_df["image_name"].nunique()}')
print(f'Label distribution (0=normal, 1=lesion):')
print(all_df['label'].value_counts().sort_index())
print()
print(all_df.groupby(['dataset_name', 'label']).size().unstack(fill_value=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

all_df['label'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Label (0=normal, 1=lesion)', color='steelblue')
axes[0].set_xticklabels(['No lesion', 'Lesion'], rotation=0)

all_df['lesion_type'].value_counts().plot(kind='bar', ax=axes[1], title='Lesion type', color='coral')
axes[1].tick_params(axis='x', rotation=30)

all_df['dataset_name'].value_counts().plot(kind='bar', ax=axes[2], title='Dataset (hospital)', color='seagreen')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/annotation_stats.png', dpi=120)
plt.show()
print('Stats chart saved.')

## 8 · Save client data to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/federated_mammography_data'
os.makedirs(DRIVE_DIR, exist_ok=True)

client_map = {
    'client1_inbreast':  CLIENT1,
    'client2_cbisddsm':  CLIENT2,
    'client3_mias':      CLIENT3,
}

for client_name, client_dir in client_map.items():
    dst = os.path.join(DRIVE_DIR, client_name)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(client_dir, dst)
    print(f'Saved {client_name} → {dst}')

# Also save the stats chart
shutil.copy(f'{DATA_DIR}/annotation_stats.png', DRIVE_DIR)
print('All client data saved to Google Drive.')

## 9 · Update config.yaml for federated training

In [ ]:
import yaml

config_path = os.path.join(PROJECT_ROOT, 'config.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['client_dirs'] = [CLIENT1, CLIENT2, CLIENT3]

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('config.yaml updated with client directories.')
for d in cfg['data']['client_dirs']:
    print(' ', d)

## Done!
Proceed to `train_federated.py` to start federated training:
```bash
!pip install flwr[simulation] torch torchvision pyyaml
!python /content/federated_project/Federated_project/train_federated.py --config config.yaml
```